In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

In [ ]:
import os
import asyncio
import time
import logging
from datetime import datetime as dt
import psutil

from rich.pretty import pprint

import panel as pn
from enderleaf.image import to_pil
from enderscope.serial import list_ports, default_printer_port, Stage
from picamera2 import Picamera2

pn.extension()

In [ ]:
mkd_log = pn.pane.Markdown(object="", sizing_mode="scale_width", height=300)
crd_log = pn.layout.Card(mkd_log, title="Log", collapsed=True, height=300)


class PanelLogHandler(logging.Handler):
    def __init__(self, target, card):
        super().__init__()
        self.target = target
        self._logs = []

    def emit(self, record):
        msg = self.format(record)
        self._logs.insert(0, msg)
        if len(self._logs) > 10:
            self._logs = self._logs[:10]
        crd_log.title = msg
        self.target.object = "- " + ("\n\n - ").join(self._logs)


class MemoryFilter(logging.Filter):

    last_process_mem = 0

    def filter(self, record):
        process: psutil.Process = psutil.Process(os.getpid())
        pmp = process.memory_percent()
        sign = (
            "⬆"
            if pmp > self.last_process_mem
            else "⬇" if pmp < self.last_process_mem else "="
        )
        record.mem_data = f"µ{sign} {pmp:02.2f}%"
        self.last_process_mem = pmp
        return True


log_file_handler = logging.FileHandler(
    os.path.join(".", f"enderleaf_{dt.now().strftime('%Y_%m_%d')}.log"),
    mode="a",
    delay=True,
)
log_file_handler.addFilter(MemoryFilter())

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s - %(mem_data)s - %(name)s - %(levelname)s] - %(message)s",
    handlers=[
        log_file_handler,
        PanelLogHandler(target=mkd_log, card=crd_log),
    ],
)

logger = logging.getLogger(os.path.splitext(__name__)[-1].replace(".", ""))
logger.info("")
logger.info(
    "============================= Starting session ============================="
)
logger.info("")

In [ ]:
camera = Picamera2()
_still_conf = camera.create_still_configuration()
_still_conf["lores"] = {"format": "BGR888", "size": (1280, 720), "preserve_ar": True}

pprint(_still_conf)

camera.start()
camera.switch_mode(_still_conf)

In [ ]:

# UI Components
progress = pn.indicators.Progress(value=0, max=100, sizing_mode="scale_width")
status = pn.widgets.StaticText(value="Ready")
bt_start = pn.widgets.Button(name="START Long Task", sizing_mode="scale_width")
bt_stop = pn.widgets.Button(
    name="STOP Long Task", align="end", sizing_mode="scale_width"
)
bt_end = pn.widgets.Button(name="END ALL", align="end", sizing_mode="scale_width")
img_rt_preview = pn.pane.Image(sizing_mode="scale_width")
img_still = pn.pane.Image(sizing_mode="scale_width")


class Controller:
    def __init__(self):
        self.request_abort = False

    async def background_task(self):
        status.value = "Running..."
        try:
            for i in range(1, 100):
                # 🔹 Replace this with your actual work
                await asyncio.sleep(0)  # Yields control to event loop

                # Simulate long process
                time.sleep(1)
                img_still.object = to_pil(camera.capture_array("main"))
                await asyncio.sleep(0)

                # 🔹 Update UI (Panel auto-pushes changes to frontend)
                progress.value = i
                status.value = f"Processing step {i}"

                # Optional: force UI flush if updates feel delayed
                await asyncio.sleep(0)

                if self.request_abort == True:
                    status.value = "❌ Abort"
                    logger.warning("❌ Abort")
                    return

        except Exception as e:
            status.value = f"❌ Error: {str(e)}"
            logger.exception(f"❌ Error: {str(e)}")
            return
        else:
            status.value = "✅ Completed!"
            logger.info("✅ Completed!")
            progress.value = 100
        finally:
            pn.state.schedule_task("preview_task", preview_task, period="0.1s")


controller = Controller()


async def preview_task():
    img_rt_preview.object = to_pil(camera.capture_array("lores"))


pn.state.schedule_task("preview_task", preview_task, period="0.1s")


def start_task(event):
    # Schedule async task without blocking the UI thread
    print(locals(), flush=True)
    logger.info("Starting task")
    controller.request_abort = False
    pn.state.cancel_task("preview_task")
    controller.color_index_step = 50
    asyncio.create_task(controller.background_task())


async def stop_task(event):
    # Stop task
    logger.info("Stopping task")
    controller.request_abort = True
    status.value = f"Abort requested by user"
    controller.color_index_step = 5
    await asyncio.sleep(0)
    pn.state.schedule_task("preview_task", preview_task, period="0.1s")


def end_all(event):
    logger.critical("Ending it all")
    pn.state.cancel_task("preview_task")


bt_start.on_click(start_task)
bt_stop.on_click(stop_task)
bt_end.on_click(end_all)

# Layout
app = pn.Column(
    pn.pane.Markdown("### Long-Running Task Demo"),
    pn.Row(img_rt_preview, img_still),
    pn.Row(bt_start, bt_stop, bt_end),
    progress,
    status,
    pn.Column(crd_log, scroll=True, height=300),
    width=400,
    align="center",
)
app.servable()

In [ ]:
for port in list_ports():
    if str(port) == str(default_printer_port()):
        _stage = Stage(port, 115200)
        _stage.safe_home()


In [ ]:
camera.capture_array("lores").shape

In [ ]:
_still_conf